# How do daily images become an annual rainfall map?

Distinguish temporal sums from spatial distributions.

**Task** · Check the daily image count for 2020, calculate the annual total and compare its spatial distribution.

Run from top to bottom. Predict the result before changing code, then compare it with the output.

## Before you start

Use a CPU runtime. You need a registered Earth Engine Cloud project and permission to use it. See the [setup guide](https://github.com/steelpl/lecture/blob/main/CE776-climate-change-carbon-water-cycle/GEE_SETUP.md). Authenticate with your own account before querying data.

In [ ]:
%pip -q install earthengine-api==1.7.52 geemap==0.39.0

In [ ]:
import ee
import geemap

PROJECT_ID = ""  # Your Cloud project ID registered for Earth Engine
if not PROJECT_ID.strip():
    raise ValueError("Enter your registered Google Cloud project ID in PROJECT_ID")
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
roi = ee.Geometry.Rectangle([126.80, 37.43, 127.15, 37.70], geodesic=False)

In [ ]:
rain = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
        .filterBounds(roi).filterDate('2020-01-01','2021-01-01').select('precipitation'))
annual = rain.sum().rename('annual_mm').clip(roi)
summary = annual.reduceRegion(ee.Reducer.mean().combine(ee.Reducer.minMax(), sharedInputs=True), roi, 5566, maxPixels=100_000)
hist = annual.reduceRegion(ee.Reducer.histogram(maxBuckets=20),roi,5566,maxPixels=100_000)

2020 is a leap year: expect 366 daily images. Summing a daily rate in mm/day over one-day intervals gives an annual total in mm. Check both the image count and the valid observation count at each pixel.

In [ ]:
print('daily images (expected 366):', rain.size().getInfo())
print('valid day counts:', rain.count().reduceRegion(ee.Reducer.minMax(),roi,5566,maxPixels=100_000).getInfo())
print('annual summary (mm):', summary.getInfo())
print('spatial histogram:', hist.getInfo())

In [ ]:
Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(annual, {'min':500,'max':2000,'palette':['white','cyan','blue']}, '2020 precipitation (mm)')
Map

Explain whether the histogram shows daily precipitation amounts or the spatial distribution of annual precipitation. Assess whether 0.05° data support neighborhood-scale interpretation in a small region near Seoul. Expand the region and compare the distribution.

## Data and interpretation

- [CHIRPS v2 Daily](https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHG_CHIRPS_DAILY)

Observation years define fixed examples, independent of the teaching year. The code demonstrates processing methods; inspect actual maps and values after authentication.